In [0]:
import time
import json
import requests
import pandas as pd

## Data Source: CQC Syndication API

For this Home/Domiciliary Care Market Analysis project, we used the **CQC Syndication API** to extract Kent-based 
domiciliary/personal care provider data.

**Base URL:** `https://api.service.cqc.org.uk/public/v1`

### Getting Access

1. Register for a free account at [api-portal.service.cqc.org.uk/signup](https://api-portal.service.cqc.org.uk/signup)
2. Subscribe to the **Syndication** product
3. Collect your subscription key from the portal's Profile page

In [0]:

BASE_URL = "https://api.service.cqc.org.uk/public/v1"

# Getting CQC subscription key / API key from the secrets in Databricks
CQC_SUBSCRIPTION_KEY = dbutils.secrets.get(
    catalog="domiciliarycare", schema="security", key="api_key"
)

HEADERS = {
    "Ocp-Apim-Subscription-Key": CQC_SUBSCRIPTION_KEY,
    "User-Agent": "HomeSafeKentPipeline/1.0",  
    "Accept": "application/json",
}
print("Confirmed - Key loaded from Unity Catalog secret.")

### API Response Handling

The `cqc_get()` function below is used to send requests to the CQC Syndication API and 
handle its response — including checking for and reacting appropriately to different 
status codes, rather than assuming every request succeeds.

Per CQC's own API documentation, requests to this endpoint can return the following 
status codes: 
- **200 (OK)**
- **400 (Bad Request)**
- **404 (Not Found)**
- **500 (Internal Server Error)**

We additionally handle :
- **401 (Unauthorized)** 
- **502 (Bad Gateway)** <br>
which are not listed in CQC endpoint documentation but occur at the API gateway level general practice. 

We also handle:
- **429 (Too Many Requests)** <br>
explicitly with a wait-and-retry mechanism, since CQC enforces a **rate limit of 100 requests per 5 seconds**. Rather than allowing the pipeline to fail when this limit is hit, our code pauses (using the `Retry-After` value 
CQC provides, or a 5-second default) and automatically retries — up to 3 attempts per 
request , so a single burst of rate limiting doesn't crash the full extraction run across hundreds of records.

In [0]:
def cqc_get(path, params=None, max_retries=3):
    url = f"{BASE_URL}{path}"
    resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code == 404:
            print(f"404 Not Found - no record exists at {url}")
            return None

        if resp.status_code == 400:
            raise RuntimeError("400 Bad Request - check your request parameters.")

        if resp.status_code == 401:
            raise RuntimeError("401 Unauthorized - key missing or invalid.")

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited (429). Waiting {wait}s before retry {attempt}/{max_retries}...")
            time.sleep(wait)
            continue

        if resp.status_code in (500, 502):
            raise RuntimeError(f"{resp.status_code} -- CQC server-side error, not a request problem.")

        resp.raise_for_status()

    raise RuntimeError(f"Gave up after {max_retries} attempts (last status {resp.status_code if resp else 'n/a'}).")

### Fetching Location Data via the CQC API

We now fetch location data through the CQC Syndication API, filtered by:

- **Local Authority** - `Kent`
- **Regulated Activity** - `Personal care`

We filter specifically on **Personal care** as Home Safe is only interested in the **home care / domiciliary care** 
market — services delivered to a person in their own home. 

In [0]:

def fetch_kent_data(local_authority="Kent", regulated_activity="Personal care", per_page=1000, polite_delay=0.3):
    """Page through /locations filtered to Kent + Personal care."""
    params = [
        ("localAuthority", local_authority),
        ("regulatedActivity", regulated_activity),
        ("perPage", per_page),
    ]
    all_locations = []
    page = 1
    while True:
        page_params = params + [("page", page)]
        data = cqc_get("/locations", params=page_params)
        all_locations.extend(data["locations"])
        print(f"Page {data['page']}/{data['totalPages']} -- {len(data['locations'])} rows "
              f"(running total {len(all_locations)})")
        if not data.get("nextPageUri"):
            break
        page += 1
        time.sleep(polite_delay)
    return all_locations

kent_locations_summary = fetch_kent_data()
print(f"\nTotal Kent personal-care locations: {len(kent_locations_summary)}")


In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")

detail_records = []
failed_ids = []
for i, loc in enumerate(kent_locations_summary, 1):
    try:
        detail_records.append(fetch_location_detail(loc["locationId"]))
    except Exception as e:
        print(f"FAILED on {loc['locationId']} ({loc.get('locationName')}): {e}")
        failed_ids.append(loc["locationId"])
    if i % 50 == 0:
        print(f"Processed {i}/{len(kent_locations_summary)}...")
    time.sleep(0.3)

print(f"\nDone. Fetched {len(detail_records)} records. Failed: {len(failed_ids)}")
if failed_ids:
    print(f"Failed IDs: {failed_ids}")